# Part 12 — Municipal units (GAUL level-2)

Load the GO+DF ADM2 municipalities from the GEE catalog (no upload) and aggregate present suitability, zones and the CMIP6 Δ per municipality with `reduceRegions` — the join table for offline rankings (Parts 14–15). Engine: `src/external.py`.

**Output:** `municipal_godf` (table asset) + a per-municipality CSV. **DoD:** GO+DF municipalities load (DF present as Brasília); `reduceRegions` runs; `ADM2_NAME` is the join key.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import ee, geemap
import utils, features
project = utils.init()
print('EE initialized; project =', project)
import external

In [ ]:
aoi = utils.load_aoi(project)
print('AOI area (km^2):', round(aoi.area(1000).divide(1e6).getInfo(), 1))

### Load the municipal units (GAUL L2, GO+DF)

In [ ]:
muni = external.municipal_fc()
print('municipalities:', muni.size().getInfo())
print('states:', muni.aggregate_array('ADM1_NAME').distinct().getInfo())

### Aggregate present suitability + zone + Δ per municipality

In [ ]:
suit = ee.Image(utils.asset_id(project, 'suit_present'))
zones = ee.Image(utils.asset_id(project, 'zones_present')).rename('zone')
delta = ee.Image(utils.asset_id(project, 'delta_ssp585_2051_2070'))
segs = list(utils.cfg('segments')['segments'])
agg_img = suit.select([f'suit_{s}' for s in segs]).addBands(zones).addBands(delta)
table = external.municipal_means(muni, agg_img)
print('reduceRegions feature count:', table.size().getInfo())

### Peek — a few municipalities' mean suitability

In [ ]:
import pandas as pd
key = utils.cfg()['gaul_l2']['join_key']
props = [key, 'ADM1_NAME'] + [f'suit_{s}' for s in segs]
rows = table.select(props, None, False).limit(8).getInfo()['features']
pd.DataFrame([f['properties'] for f in rows]).round(3)

### Export — `municipal_godf` table asset (with geometry) + municipal CSV to Drive

In [ ]:
utils.ensure_folder(project)
# asset export keeps geometry (Export.table.toAsset rejects null geometry);
# the CSV drops it via property selectors for a light offline join.
t1 = utils.export_table(table, project, 'municipal_godf')
csv_cols = [key, 'ADM1_NAME'] + [f'suit_{s}' for s in segs] + ['zone']
t2 = ee.batch.Export.table.toDrive(collection=table.select(csv_cols, None, False),
        description='municipal_suit_summary', fileFormat='CSV')
t2.start()
print(utils.task_summary([t1, t2]))